In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 8 - WEEK 9 BAYESIAN OPTIMISATION
# Run from inside the week9/ folder
# ============================================================

# ------------------------------------------------------------
# 1. Load cumulative Week 9 data
# ------------------------------------------------------------

X = np.load("function8/initial_inputs.npy")
Y = np.load("function8/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 8 calibration check
# ------------------------------------------------------------
#
# Week 8 selected:
# [0.10748302, 0.10246352, 0.11540771, 0.18193642,
#  0.81284469, 0.61082360, 0.14921134, 0.60689407]
#
# GP prediction:
# mean ≈ 9.992824
# std  ≈ 0.047989
#
# Actual:
# 9.9784413968839
# ------------------------------------------------------------

week8_pred_mean = 9.992824
week8_pred_std = 0.047989
week8_actual = 9.9784413968839

week8_error = week8_actual - week8_pred_mean
week8_z_error = week8_error / week8_pred_std

print("\n================================")
print("WEEK 8 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week8_pred_mean)
print("Predicted std :", week8_pred_std)
print("Actual        :", week8_actual)

print("\nPrediction error:")
print(week8_error)

print("\nError / predicted std:")
print(week8_z_error)


# ------------------------------------------------------------
# 3. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(8) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales
relative_sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:")
print(lengthscales)

print("\nNormalised inverse-lengthscale sensitivity:")
print(relative_sensitivity)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(mu, sigma, best_y, xi=0.0):

    improvement = mu - best_y - xi

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)
    Z[valid] = improvement[valid] / sigma[valid]

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        +
        sigma[valid] * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Candidate generation
# ------------------------------------------------------------

rng = np.random.default_rng(42)

local_scale = np.clip(
    0.25 * lengthscales,
    0.015,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.04,
    0.20
)

print("\nLocal widths:")
print(local_scale)

print("\nWide widths:")
print(wide_scale)

local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(150000, 8)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(100000, 8)
    )
)

global_candidates = rng.uniform(
    0,
    1,
    size=(200000, 8)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])


# ------------------------------------------------------------
# 6. Remove near-duplicates
# ------------------------------------------------------------

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 7. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 8. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 9. EI sensitivity
# ------------------------------------------------------------

y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\n================================")
print("EI SENSITIVITY")
print("================================\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


# ------------------------------------------------------------
# 10. Highest predicted mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


# ------------------------------------------------------------
# 11. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [0.1, 0.25, 0.5, 1.0]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )

X shape: (48, 8)
Y shape: (48,)

Current best:
[0.095473 0.144081 0.145356 0.087993 0.834791 0.549036 0.181794 0.551605] -> 9.991464829113

Y range:
min = 5.5921933895401965
max = 9.991464829113
std = 1.171511483457472

WEEK 8 CALIBRATION CHECK
Predicted mean: 9.992824
Predicted std : 0.047989
Actual        : 9.9784413968839

Prediction error:
-0.014382603116100512

Error / predicted std:
-0.29970624760050246


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/co


GP FIT

Fitted kernel:
0.898**2 * Matern(length_scale=[1.26, 2, 0.988, 2, 2, 2, 1.36, 2], nu=2.5) + WhiteKernel(noise_level=1e-08)

ARD lengthscales:
[1.25502751 2.         0.98778153 2.         2.         2.
 1.35773085 2.        ]

Normalised inverse-lengthscale sensitivity:
[0.15791609 0.09909452 0.20064055 0.09909452 0.09909452 0.09909452
 0.14597078 0.09909452]

Local widths:
[0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]

Wide widths:
[0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2]

Candidates after duplicate filtering:
449998

PRIMARY EI
candidate = [0.         0.07667186 0.         0.07419133 0.80525627 0.17859206
 0.18260923 0.41706558]
mean = 9.86036459182207
std = 0.2180638697756279
EI = 0.03670964999740843

EI SENSITIVITY

xi = 0.000000e+00 
 candidate = [0.         0.07667186 0.         0.07419133 0.80525627 0.17859206
 0.18260923 0.41706558] 
 mean = 9.860365 
 std = 0.218064 
 EI = 0.03670965 

xi = 1.171511e-02 
 candidate = [0.         0.07667186 0.         0.07419133 0.80525627 0.17859206
 0.1

In [2]:
# ============================================================
# FINAL FUNCTION 8 - WEEK 9 SELECTION
# ============================================================
#
# Week 8 calibration error was only about -0.30 sigma,
# so the local GP remains reasonably trustworthy.
#
# EI is rejected because it moves toward boundary-heavy,
# high-uncertainty points with substantially lower mean.
#
# beta = 0.25 gives modest exploration while remaining
# in the established high-performing basin.

beta = 0.25

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week9_candidate = candidates[final_idx]

print("Week 9 Function 8 candidate:")
print(week9_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nUCB:")
print(UCB[final_idx])

portal = "-".join(
    f"{x:.6f}"
    for x in week9_candidate
)

print("\nPortal format:")
print(portal)

Week 9 Function 8 candidate:
[0.11706245 0.1831352  0.10430401 0.12041429 0.77064421 0.49765153
 0.17958835 0.55885207]

Predicted mean:
9.996568618609157

Predicted std:
0.037622447513213135

UCB:
10.00597423048746

Portal format:
0.117062-0.183135-0.104304-0.120414-0.770644-0.497652-0.179588-0.558852
